# Загрузка данных и библиотек
---

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import seaborn as sns
import copy

RANDOM_STATE = 0xDEF

In [ ]:
train_raw = pd.read_csv('./data/train.csv')
test_raw = pd.read_csv('./data/test.csv')

train = train_raw.copy()
test = test_raw.copy()

# EDA
---

In [ ]:
train.head()

Посмотрим на распределение таргета `SalePrice`

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(train['SalePrice'], bins=50, edgecolor='black')

plt.title('Гистограмма распределения SalePrice')
plt.xlabel('Значение')
plt.ylabel('Частота')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

Видим, что распределение нессиметрично, попробуем взять log от величины

In [ ]:
train['SalePrice_log'] = np.log1p(train['SalePrice'])

plt.figure(figsize=(8, 5))
plt.hist(train['SalePrice_log'], bins=50, edgecolor='black')

plt.title('Гистограмма распределения SalePrice')
plt.xlabel('Значение')
plt.ylabel('Частота')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

График стал более симметричным и похожим на нормальное распределение

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(train['SalePrice_log'], vert=False, patch_artist=True)
plt.title('Поиск выбросов в ценах на дома')
plt.xlabel('Цена')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Расчет Z-score для каждой цены
train['Z_Score'] = (train['SalePrice_log'] - train['SalePrice_log'].mean()) / train['SalePrice_log'].std()

# Находим выбросы (где Z-score больше 3 или меньше -3)
outliers_z = train[train['Z_Score'].abs() > 3]
print("Выбросы по методу Z-score:")
print(outliers_z[['SalePrice_log']])


In [ ]:
outliers_z[['SalePrice', 'Neighborhood', 'SaleCondition']].sort_values(by='SalePrice')

### Анализ пропусков

In [ ]:
with pd.option_context('display.max_rows', None):
    print(train.isna().sum()[train.isna().sum() > 0].sort_values())

Рассмотрим пропуски для `PoolQC`, вполне возможно пропуски означают отсутствие объекта

In [ ]:
print((train[train['PoolQC'].isna()].index != train[train['PoolArea'] == 0].index).sum())

ВЫВОД: пропуски в `PoolQC` означают отсутствие объекта

In [ ]:
train['MiscFeature'].unique()

# Предобработка данных
---

In [ ]:
train = train_raw.copy()
test  = test_raw.copy()

df = pd.concat([train, test], sort=False).reset_index(drop=True)
train_mask = df['SalePrice'].notna()

### Заполнение пропусков

In [ ]:
total = df.isna().sum()
percent = np.round((df.isna().sum() / df.shape[0]) * 100, 2)

missing_df = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_df.drop('SalePrice', inplace=True)
display(missing_df[missing_df['Total'] > 0].sort_values(by='Total', ascending=False))

Для категориальных признаков отсутствие объекта вместо `NaN` заполним строкой `"None"`

In [ ]:
none_columns_categorical = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType',
]
for col in none_columns_categorical:
    df[col] = df[col].fillna('None')

А для числовых признаков - заполним нулём

In [ ]:
none_columns_numeric = [
    'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea',
]
for col in none_columns_numeric:
    df[col] = df[col].fillna(0)

Пропуски `GarageYrBlt` можно заменить годом постройки дома `YearBuilt`

In [ ]:
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])

Пропуски `LotFrontage` можно заменить медианой внутри `Neighborhood`

In [ ]:
df.loc[train_mask, 'LotFrontage'] = df[train_mask].groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
df.loc[~train_mask, 'LotFrontage'] = df[~train_mask].groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

Оставшиеся единичные пропуски можно заменить модой

In [ ]:
mode_fill_columns = ['MSZoning', 'Utilities', 'Functional', 'Exterior1st', 'Exterior2nd', 'Electrical', 'KitchenQual', 'SaleType']
for col in mode_fill_columns:
    train_mode = df.loc[train_mask, col].mode()[0]
    df.loc[train_mask, col] = df.loc[train_mask, col].fillna(train_mode)
    test_mode = df.loc[~train_mask, col].mode()[0]
    df.loc[~train_mask, col] = df.loc[~train_mask, col].fillna(test_mode)

### Преобразования

Переведем `MSSubClass` в категориальный признак

In [ ]:
df['MSSubClass'] = df['MSSubClass'].astype('category')

Рассмотрим статистику по категориальным признакам

In [ ]:
categorical_stats = df.describe(include=['object', 'category'])

with pd.option_context('display.max_columns', None):
    display(categorical_stats)

In [ ]:
exclude_columns = ['Id', 'SalePrice']

THRESHOLD = 6

num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_features = [col for col in num_features if col not in exclude_columns]

cat_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_columns = [col for col in cat_columns if col not in exclude_columns]

ohe_features = []
target_features = []

for col in cat_columns:
    n_unique = df.loc[train_mask, col].nunique()
    if n_unique <= THRESHOLD:
        ohe_features.append(col)
    else:
        target_features.append(col)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def target_encode_cv(df, train_mask, target_features, y, cv, smoothing=10):
    """
    Честный (без утечки) target encoding для категориальных признаков высокой кардинальности.

    Train: каждая строка получает OOF-закодированное значение — среднее по таргету
    внутри категории, посчитанное БЕЗ фолда, в который попала сама строка.
    Test: кодируется средним по категории, посчитанным на всём train.

    smoothing сглаживает редкие категории к глобальному среднему:
    encoded = (count * category_mean + smoothing * global_mean) / (count + smoothing)
    """
    df = df.copy()
    global_mean = y.mean()
    train_idx = df.index[train_mask]
    test_idx = df.index[~train_mask]

    for col in target_features:
        encoded = pd.Series(index=df.index, dtype=float)

        # --- Train: честные OOF-значения ---
        for tr_pos, val_pos in cv.split(train_idx):
            fold_train_idx = train_idx[tr_pos]
            fold_val_idx = train_idx[val_pos]

            stats = pd.DataFrame({'y': y.loc[fold_train_idx], 'cat': df.loc[fold_train_idx, col]}) \
                        .groupby('cat')['y'].agg(['mean', 'count'])
            smoothed = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)

            encoded.loc[fold_val_idx] = df.loc[fold_val_idx, col].map(smoothed)

        # категории, не встретившиеся в конкретном обучающем фолде, — глобальным средним
        encoded.loc[train_idx] = encoded.loc[train_idx].fillna(global_mean)

        # --- Test: кодируем средним по всему train ---
        stats_full = pd.DataFrame({'y': y, 'cat': df.loc[train_idx, col]}).groupby('cat')['y'].agg(['mean', 'count'])
        smoothed_full = (stats_full['count'] * stats_full['mean'] + smoothing * global_mean) / (stats_full['count'] + smoothing)
        encoded.loc[test_idx] = df.loc[test_idx, col].map(smoothed_full).fillna(global_mean)

        df[col + '_te'] = encoded

    return df.drop(columns=target_features)

In [ ]:
df_cat = df.copy()
df = pd.get_dummies(df, columns=ohe_features)
df = target_encode_cv(df, train_mask, target_features, y=df.loc[train_mask, 'SalePrice'], cv=cv, smoothing=10)

In [ ]:
train, test = df[train_mask], df[~train_mask]
train_cat, test_cat = df_cat[train_mask], df_cat[~train_mask]
test.drop(['SalePrice'], axis=1, inplace=True)
test_cat.drop(['SalePrice'], axis=1, inplace=True)

X = train.drop('SalePrice', axis=1)
y = train['SalePrice']
X_cat = train_cat.drop('SalePrice', axis=1)
y_cat = train_cat['SalePrice']

# Classic ML
---

Функция для обучения моделей

In [ ]:
def run_model(name, estimator, X, y, cv, needs_scaling=False, fit_params=None, stats=None, log_target=True):
    """
    Честная OOF-оценка + финальная модель на 100% данных.

    log_target=True: модель обучается предсказывать log1p(SalePrice), а не саму цену.
    Именно так считается метрика конкурса House Prices — RMSE между логарифмами
    предсказанной и настоящей цены, поэтому обучение и honest OOF-оценка идут
    напрямую в лог-пространстве, без обратного преобразования.
    Дополнительно печатается RMSE в долларах — только для интуиции/отчёта,
    в stats и в сравнение моделей эта цифра не идёт.
    """
    fit_params = fit_params or {}
    y_model = np.log1p(y) if log_target else y
    oof_preds = np.zeros(len(y))
    fold_scores = []

    for train_idx, val_idx in cv.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_model.iloc[train_idx], y_model.iloc[val_idx]

        if needs_scaling:
            scaler = StandardScaler()
            X_tr, X_val = scaler.fit_transform(X_tr), scaler.transform(X_val)

        model = clone(estimator)
        model.fit(X_tr, y_tr, **fit_params)

        fold_pred = model.predict(X_val)
        oof_preds[val_idx] = fold_pred
        fold_scores.append(root_mean_squared_error(y_val, fold_pred))

    total_rmse = root_mean_squared_error(y_model, oof_preds)
    if stats is not None:
        stats[name] = total_rmse

    msg = f'{name}: OOF RMSE(log) = {total_rmse:.4f} | по фолдам: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}'
    if log_target:
        rmse_dollars = root_mean_squared_error(y, np.expm1(oof_preds))
        msg += f' | справочно, RMSE($): {rmse_dollars:,.0f}'
    print(msg)

    scaler_full = StandardScaler().fit(X) if needs_scaling else None
    X_full = scaler_full.transform(X) if scaler_full else X
    final_model = clone(estimator).fit(X_full, y_model, **fit_params)

    return {
        'oof_preds': oof_preds, 'model': final_model, 'scaler': scaler_full,
        'fold_scores': fold_scores, 'log_target': log_target,
    }

### Dummy Regressor - baseline

In [ ]:
y_log = np.log1p(y)

dummy_model = DummyRegressor(strategy='mean')
dummy_model.fit(X, y_log)
preds_log = dummy_model.predict(X)

rmse_log = root_mean_squared_error(y_log, preds_log)
rmse_dollars = root_mean_squared_error(y, np.expm1(preds_log))
print(f"Dummy Regressor: RMSE(log) = {rmse_log:.4f} | справочно, RMSE($) = {rmse_dollars:_.4f}")

stats = {'DummyRegressor': rmse_log}

### Linear Regression

In [ ]:
def get_tuned_linear_regression(X, y, cv, verbose=True):
    """Качественный тюнинг линейной регрессии (ElasticNet) через GridSearchCV."""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNet(max_iter=5000, random_state=RANDOM_STATE))
    ])
    
    param_grid = {
        'model__alpha': np.logspace(-4, 2, 15),
        'model__l1_ratio': [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
    }

    search = GridSearchCV(
        pipeline, 
        param_grid, 
        cv=cv, 
        scoring='neg_mean_squared_error', 
        n_jobs=-1
    )
    
    search.fit(X, y)
    
    if verbose:
        print('Linear Regression (ElasticNet) tuned:')
        # Выводим лучшие параметры в вашем фирменном стиле
        print(pd.DataFrame.from_dict(search.best_params_, orient='index', columns=['value']), end='\n')
        # Дополнительно выведем лучший корень из MSE на кросс-валидации
        best_rmse = np.sqrt(-search.best_score_)
        print(f"Best CV RMSE: {best_rmse:.4f}\n")
        
    return search.best_estimator_.named_steps['model']


# Запуск тюнинга (используется созданный ранее KFold cv для регрессии)
lr_tuned = get_tuned_linear_regression(X, y, cv)

In [ ]:
stats_stage1 = {}
results_stage1 = {}

results_stage1['lr'] = run_model('LinearRegression', lr_tuned, X, y, cv, needs_scaling=True, stats=stats, log_target=True)
lr_model = results_stage1['lr']['model']

### KNN